In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [ ]:
import time
import io
from threading import Condition

from rich.pretty import pprint

from picamera2.encoders import JpegEncoder
from picamera2.outputs import FileOutput
from picamera2 import Picamera2, Preview
from libcamera import Transform, controls

import panel as pn

from enderleaf.image import Rectangle

In [ ]:
a = Rectangle(0, 100, 0, 100)
pprint(a)
pprint(a.shrink(10, 10))

In [ ]:
pn.extension()

In [ ]:
class StreamingOutput(io.BufferedIOBase):
    def __init__(self, panel_output):
        # self.frame = None
        # self.condition = Condition()
        self.panel_output = panel_output

    def write(self, buf):
        # print("tttttt")
        self.panel_output.object = buf
        # with self.condition:
        #     self.frame = buf
        #     self.condition.notify_all()

In [ ]:
pane = pn.pane.Placeholder("toto", width=960, height=540)
pane

In [ ]:
# with Picamera2() as picam2:
picam2 = Picamera2()
pprint(picam2.sensor_resolution)
mode = picam2.sensor_modes[2]
pprint(mode)
config = picam2.create_video_configuration(
    main={"size": mode["size"]}, lores={"size": (320, 240)}, encode="main"
)
picam2.configure(config)
picam2.start_recording(JpegEncoder(), FileOutput(StreamingOutput(panel_output=pane)))
picam2.autofocus_cycle()
time.sleep(4)
new_size = 1000
window = (
    Rectangle(top=0, left=0, bottom=2592, right=4608)
    .shrink(
        new_width=new_size, new_height=new_size  # * mode["size"][1] / mode["size"][0]
    )
    .ensure_int()
)
pprint(window)
picam2.set_controls(
    {"ScalerCrop": (window.left, window.right, window.width, window.height)}
)
pprint(picam2.controls)
pprint(config)
time.sleep(3)
picam2.stop_encoder()
picam2.close()

In [ ]:
picam2.close()
picam2 = None